# GENA-LM Sequence classification
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AIRI-Institute/GENA_LM/blob/main/notebooks/GENA_sequence_classification_example.ipynb)

In [ ]:


!nvcc --version
!nvidia-smi
import torch
print("Torch CUDA Version:", torch.version.cuda)
print("CUDA Available:", torch.cuda.is_available())
!pip install scikit-learn matplotlib seaborn
!pip uninstall pydantic -y
!pip cache purge
!pip install "pydantic>=2.0"

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Wed_Nov_22_10:17:15_PST_2023
Cuda compilation tools, release 12.3, V12.3.107
Build cuda_12.3.r12.3/compiler.33567101_0


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Mon May 26 03:52:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-SXM2-16GB           On  |   00000000:07:00.0 Off |                    0 |
| N/A   38C    P0             42W /  300W |    1231MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('AIRI-Institute/gena-lm-bert-base-athaliana', force_download=True)
model = AutoModel.from_pretrained('AIRI-Institute/gena-lm-bert-base-athaliana', trust_remote_code=True, revision="main", force_download=True)

gena_module_name = model.__class__.__module__

import importlib
cls = getattr(importlib.import_module(gena_module_name), 'BertForSequenceClassification')
cls

import sklearn
from datasets import load_dataset
from datasets import Dataset
import pandas as pd

model = cls.from_pretrained('AIRI-Institute/gena-lm-bert-base-athaliana', num_labels=2)

tokenizer_config.json:   0%|          | 0.00/46.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

modeling_bert.py:   0%|          | 0.00/97.1k [00:00<?, ?B/s]

2025-06-12 05:37:50.155555: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749706670.177346      46 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749706670.184007      46 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-12 05:37:50.206797: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


pytorch_model.bin:   0%|          | 0.00/541M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at AIRI-Institute/gena-lm-bert-base-athaliana and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fine-tune model

In [ ]:
import pandas as pd
import numpy as np
import shutil
import torch
import random
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import seaborn
import seaborn as sns


from transformers import (
    AutoTokenizer, AutoModel, TrainingArguments, Trainer,
    DataCollatorWithPadding
)
import importlib
import os


# Initialize tokenizer and model name
MODEL_NAME = 'AIRI-Institute/gena-lm-bert-base-athaliana'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def balance_dataset(df):
    class_0 = df[df['label'] == 0]
    class_1 = df[df['label'] == 1]
    if len(class_0) > len(class_1):
        class_0 = resample(class_0, replace=False, n_samples=len(class_1), random_state=42)
    else:
        class_1 = resample(class_1, replace=False, n_samples=len(class_0), random_state=42)
    return pd.concat([class_0, class_1]).sample(frac=1, random_state=42).reset_index(drop=True)

def prepare_dataset(df, random_seed):
    df = df[['sequence', 'peak', 'gene']].rename(columns={'sequence': 'text', 'peak': 'label'})
    train_df, temp_df = train_test_split(df, test_size=0.7, random_state=random_seed, stratify=df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.7, random_state=random_seed, stratify=temp_df['label'])
    train_df = balance_dataset(train_df)
    val_df = balance_dataset(val_df)
    test_df = balance_dataset(test_df)
    return DatasetDict({
        'train': Dataset.from_pandas(train_df),
        'val': Dataset.from_pandas(val_df),
        'test': Dataset.from_pandas(test_df)
    })

def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {'accuracy': (predictions == labels).sum() / len(labels)}



def plot_confusion_matrix(true_labels, pred_labels, run_id):
    cm = confusion_matrix(true_labels, pred_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f'Confusion Matrix:  Run {run_id}')
    plt.savefig(f"predictions_d300_L3/confusion_matrix_run{run_id}.png")
    plt.close()

def plot_roc_auc(true_labels, probs, run_id):
    fpr, tpr, _ = roc_curve(true_labels, probs[:, 1])
    roc_auc = auc(fpr, tpr)
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve: Run {run_id}')
    plt.legend(loc="lower right")
    plt.savefig(f"predictions_d300_L3/roc_auc_run{run_id}.png")
    plt.close()

def run_single_experiment(df, run_id, seed):
    random_seed = seed + run_id
    dataset = prepare_dataset(df, random_seed=random_seed)
    tokenized_dataset = dataset.map(preprocess_function, batched=True)

    model_class = getattr(importlib.import_module(AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True).__class__.__module__), 'BertForSequenceClassification')
    model = model_class.from_pretrained(MODEL_NAME, num_labels=2)

    training_args = TrainingArguments(
        output_dir=f"output_run_{run_id}",
        learning_rate=1.35e-06,
        lr_scheduler_type="constant_with_warmup",
        warmup_ratio=0.1,
        optim='adamw_torch',
        weight_decay=0.0,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=5,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        seed=seed,
        dataloader_num_workers=0,
        metric_for_best_model="accuracy",
        greater_is_better=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["val"],
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics
    )

    trainer.train()
    log_history = trainer.state.log_history
    for split in ['train', 'val', 'test']:
        raw_split = dataset[split]
        raw_df = pd.DataFrame(raw_split)
        raw_df.to_csv(f"predictions_d300_L3/split_data_{run_id}_{split}.txt", sep="\t", index=False)

    pd.DataFrame(log_history).to_csv(f"predictions_d300_L3/training_log_{run_id}.txt", sep="\t", index=False)

    predictions = trainer.predict(tokenized_dataset['test'])
    logits = predictions.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    predicted_labels = np.argmax(logits, axis=1)
    true_labels = predictions.label_ids
    gene_names = tokenized_dataset['test']['gene']



    # Plot and save confusion matrix and ROC-AUC
    plot_confusion_matrix(true_labels, predicted_labels, run_id)
    plot_roc_auc(true_labels, probs, run_id)



    with open(f"predictions_d300_L3/predictions_run_{run_id}.txt", "w") as f:
        f.write("gene\ttrue_label\tpredicted_label\tprob_0\tprob_1\n")
        for i in range(len(gene_names)):
            f.write(f"{gene_names[i]}\t{true_labels[i]}\t{predicted_labels[i]}\t{probs[i][0]:.6f}\t{probs[i][1]:.6f}\n")

    accuracy = compute_metrics((logits, true_labels))['accuracy']
    print(f"✅ Run {run_id} Accuracy: {accuracy:.4f}")
    model.save_pretrained(f"saved_models/best_d300_L3_model_run{run_id}")
    tokenizer.save_pretrained(f"saved_models/best_d300_L3_model_run{run_id}")
    run_output_dir = f"output_run_{run_id}"
    if os.path.exists(run_output_dir):
        shutil.rmtree(run_output_dir)
    return accuracy


accs = []
# Load your data
df = pd.read_table("file/downstream_300_seq_Nointron.txt")

# Ensure predictions_all directory exists
os.makedirs("predictions_d300_L3", exist_ok=True)

accuracies = []
for run in range(7):
    acc = run_single_experiment(df, run_id=run + 1, seed=1550)
    accuracies.append(acc)
    accs.append(acc)
    results_summary = accs


print("✅ All Accuracies:", accuracies)
print(f"📊 Mean Accuracy: {np.mean(accuracies):.4f}")








In [ ]:

base_model = AutoModel.from_pretrained(
    'AIRI-Institute/gena-lm-bert-base-athaliana',
    trust_remote_code=True,
    revision="main"
)
gena_module_name = base_model.__class__.__module__
cls = getattr(importlib.import_module(gena_module_name), "BertForSequenceClassification")


def calculate_importance(dataset, model, tokenizer, output_file):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    special_tokens = {
        tokenizer.cls_token,
        tokenizer.sep_token,
        tokenizer.pad_token,
        tokenizer.eos_token,
        tokenizer.bos_token,
        None
    }

    with open(output_file, "w") as file:

        file.write("Gene\t" + "\t".join([f"Region {i}" for i in range(1, 6)]) + "\n")

        for seq, gene_name in zip(dataset["text"], dataset["gene"]):


            inputs = tokenizer(
                seq,
                return_tensors="pt",
                truncation=True,
                padding="max_length",
                max_length=128,
                return_offsets_mapping=True
            )

            inputs.pop("offset_mapping")
            input_ids = inputs["input_ids"].to(device)
            attention_mask = inputs["attention_mask"].to(device)

            embedding_layer = model.bert.embeddings
            input_embeds = embedding_layer(input_ids)
            input_embeds.retain_grad()

            outputs = model(inputs_embeds=input_embeds, attention_mask=attention_mask)
            logits = outputs.logits
            probs = softmax(logits, dim=-1)
            pred_class = torch.argmax(probs, dim=-1)

            loss = torch.nn.CrossEntropyLoss()(logits, pred_class)
            loss.backward()

            grads = input_embeds.grad
            token_importance = grads.abs().sum(dim=-1).squeeze().tolist()
            tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())

            clean_importance = [
                imp for t, imp in zip(tokens, token_importance) if t not in special_tokens
            ]

            token_importance = clean_importance

            # ---- Split into 5 regions ----
            REGION_COUNT = 5
            region_len = len(token_importance) // REGION_COUNT
            region_scores = []

            for i in range(REGION_COUNT):
                start = i * region_len
                end = (i + 1) * region_len if i < REGION_COUNT - 1 else len(token_importance)
                region_scores.append(sum(token_importance[start:end]))

            # Normalize
            total = sum(region_scores)
            region_scores = [0 if total == 0 else r / total for r in region_scores]

            file.write(f"{gene_name}\t" + "\t".join(f"{r:.4f}" for r in region_scores) + "\n")



df = pd.read_table("file/downstream_300_seq_Nointron.txt")
df = df[["sequence", "peak", "gene"]].rename(columns={"sequence": "text", "peak": "label"})
dataset = Dataset.from_pandas(df)

# --------------------------------------------------------------------
# Run models run1–run7
# --------------------------------------------------------------------
saved_model_dir = "/dicos_ui_home/tzuhsiang/test/organize/saved_models"

for run_id in range(1, 8):
    model_name = f"best_d300_L3_model_run{run_id}"
    model_path = os.path.join(saved_model_dir, model_name)

    if not os.path.isdir(model_path):
        print(f"⚠️ Model not found: {model_path}")
        continue

    print(f"🔍 Running importance for model: {model_name}")

    # Load model
    model = cls.from_pretrained(model_path)

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        'AIRI-Institute/gena-lm-bert-base-athaliana',
        trust_remote_code=True
    )

    # Output directory
    os.makedirs("predictions_d300_L3", exist_ok=True)
    output_file = f"predictions_d300_L3/token_importance_region5_{model_name}.txt"

    calculate_importance(dataset, model, tokenizer, output_file)


🔍 Running importance for model: best_d300_L3_model_run1


/dicos_ui_home/tzuhsiang/.local/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/dicos_ui_home/tzuhsiang/.local/lib/python3.12/site-packages/transformers/modeling_utils.py:1126: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


🔍 Running importance for model: best_d300_L3_model_run2
🔍 Running importance for model: best_d300_L3_model_run3
🔍 Running importance for model: best_d300_L3_model_run4
🔍 Running importance for model: best_d300_L3_model_run5
🔍 Running importance for model: best_d300_L3_model_run6
🔍 Running importance for model: best_d300_L3_model_run7


predict human

In [ ]:
import pandas as pd
import torch
import os
import importlib
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score

# Settings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_file = "file/input_human_seq300.txt"  # Update path as needed
output_dir = "predictions_human_L3"
os.makedirs(output_dir, exist_ok=True)

# Load dataset
df = pd.read_table(input_file)  # expects: sequence, peak, gene

# Model directories
model_dirs = [f"saved_models/best_d300_L3_model_run{i}" for i in range(1, 9)]

# Loop over models
for i, model_dir in enumerate(model_dirs, start=1):
    print(f"🔍 Running predictions with {model_dir}")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_dir)

    # Load model class via importlib
    base_model = AutoModel.from_pretrained(model_dir)
    gena_module_name = base_model.__class__.__module__
    cls = getattr(importlib.import_module(gena_module_name), 'BertForSequenceClassification')
    model = cls.from_pretrained(model_dir).to(device).eval()

    # Track predictions and labels
    all_preds = []
    all_labels = []

    output_file = os.path.join(output_dir, f"predictions_run{i}.txt")
    with open(output_file, "w") as f:
        for idx, row in df.iterrows():
            x, y, gene = row['sequence'], row['peak'], row['gene']

            # Tokenize
            x_feat = tokenizer(x, return_tensors='pt', truncation=True, padding=True, max_length=128)
            x_feat = {k: v.to(device) for k, v in x_feat.items()}

            # Predict
            with torch.no_grad():
                out = model(**x_feat)
                prob = torch.softmax(out.logits, dim=-1)
                pred_label = torch.argmax(prob, dim=1).item()

            all_preds.append(pred_label)
            all_labels.append(y)

            # Save to file
            prob_values = prob.squeeze().cpu().numpy()
            f.write(f"Gene: {gene}\tPrediction: {pred_label}\tTrue Label: {y}\tProbabilities: {prob_values.tolist()}\n")

    # Accuracy
    acc = accuracy_score(all_labels, all_preds)
    print(f"✅ Run {i} Accuracy: {acc:.4f}")


TRAM1 delete


In [ ]:

test_human_TRAM_three = pd.read_table("file/input_human_TRAM1_kmers_12.txt")

test_human_TRAM_three.to_csv('test_human_TRAM1_kmers_12.csv', index=False, columns=['sequence', 'peak','gene'])

# Load the datasets from the CSV files
dataset = load_dataset('csv', data_files={  'test_human_TRAM_three': 'test_human_TRAM1_kmers_12.csv'
                        })


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load base model to determine the correct class
base_model_path = 'AIRI-Institute/gena-lm-bert-base-athaliana'
tokenizer_base = AutoTokenizer.from_pretrained(base_model_path)
temp_model = AutoModel.from_pretrained(base_model_path, trust_remote_code=True)
gena_module_name = temp_model.__class__.__module__
cls = getattr(importlib.import_module(gena_module_name), 'BertForSequenceClassification')



# Loop over 7 models
for run_num in range(1, 8):
    print(f"\n===== Running prediction with model run {run_num} =====")
    model_path = f"saved_models/best_d300_L3_model_run{run_num}/"
    output_file = f"predictions_TRAM_run{run_num}.txt"

    # Load model and tokenizer for the current run
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = cls.from_pretrained(model_path, num_labels=2).to(device).eval()

    all_preds = []
    all_labels = []

    with open(output_file, "w") as file:
        for i in range(len(test_dataset[dataset_name])):
            x = test_dataset[dataset_name]['sequence'][i]
            y = test_dataset[dataset_name]['peak'][i]
            gene = test_dataset[dataset_name]['gene'][i]

            # Tokenize input
            x_feat = tokenizer(x, return_tensors='pt', truncation=True, padding=True, max_length=256)
            x_feat = {k: v.to(device) for k, v in x_feat.items()}

            # Inference
            with torch.no_grad():
                out = model(**x_feat)
                prob = torch.softmax(out.logits, dim=-1)
                pred_label = torch.argmax(prob, dim=1).item()

            all_preds.append(pred_label)
            all_labels.append(y)

            # Log prediction to file
            file.write(f'Gene: {gene}, Prediction: {pred_label}, True Label: {y}, Probabilities: {prob.cpu().numpy()}\n')

    # Compute accuracy
    accuracy = accuracy_score(all_labels, all_preds)
    print(f'Run {run_num} Accuracy: {accuracy:.4f}')